In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Environment ready ✅")


Environment ready ✅


FileNotFoundError: [Errno 2] No such file or directory: '../data/movies.csv'

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (437827765.py, line 1)

In [4]:
movies = pd.read_csv("C:\Users\VSB-AIDSPC15\Downloads\Movie Recommendation Engine\movies.csv")

movies.head()


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (4239552240.py, line 1)

In [5]:
movies = pd.read_csv("../data/movies.csv")
movies.head()


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [6]:
# Fill missing genres (if any)
movies['genres'] = movies['genres'].fillna('')

# Replace | with space and convert to lowercase
movies['genres_clean'] = (
    movies['genres']
    .str.replace('|', ' ', regex=False)
    .str.lower()
)

movies[['title', 'genres', 'genres_clean']].head()


KeyError: 'genres'

In [7]:
movies.info()


<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  4803 non-null   int64
 1   title     4803 non-null   str  
 2   cast      4803 non-null   str  
 3   crew      4803 non-null   str  
dtypes: int64(1), str(3)
memory usage: 150.2 KB


In [8]:
# Combine cast and crew into one feature
movies['content'] = movies['cast'] + ' ' + movies['crew']

movies[['title', 'content']].head()


,title,content
0,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""..."
1,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa..."
2,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr..."
3,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba..."
4,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c..."


In [9]:
movies['content'] = (
    movies['content']
    .str.lower()
    .str.replace(r'[^a-zA-Z\s]', '', regex=True)
)

movies['content'].head()


0    castid  character jake sully creditid aacaca g...
1    castid  character captain jack sparrow crediti...
2    castid  character james bond creditid fedcaedd...
3    castid  character bruce wayne  batman creditid...
4    castid  character john carter creditid feacafe...
Name: content, dtype: str

In [10]:
import ast

In [11]:
def extract_names(text, key='name', top_n=3):
    try:
        data = ast.literal_eval(text)
        names = [item[key] for item in data][:top_n]
        return ' '.join(names)
    except:
        return ''


In [12]:
movies['actors'] = movies['cast'].apply(lambda x: extract_names(x, top_n=3))
movies[['title', 'actors']].head()


,title,actors
0,Avatar,Sam Worthington Zoe Saldana Sigourney Weaver
1,Pirates of the Caribbean: At World's End,Johnny Depp Orlando Bloom Keira Knightley
2,Spectre,Daniel Craig Christoph Waltz Léa Seydoux
3,The Dark Knight Rises,Christian Bale Michael Caine Gary Oldman
4,John Carter,Taylor Kitsch Lynn Collins Samantha Morton


In [13]:
def get_director(text):
    try:
        data = ast.literal_eval(text)
        for item in data:
            if item.get('job') == 'Director':
                return item.get('name')
        return ''
    except:
        return ''

movies['director'] = movies['crew'].apply(get_director)
movies[['title', 'director']].head()


,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [14]:
movies['content'] = (
    movies['actors'] + ' ' + movies['director']
).str.lower()

movies[['title', 'content']].head()


,title,content
0,Avatar,sam worthington zoe saldana sigourney weaver j...
1,Pirates of the Caribbean: At World's End,johnny depp orlando bloom keira knightley gore...
2,Spectre,daniel craig christoph waltz léa seydoux sam m...
3,The Dark Knight Rises,christian bale michael caine gary oldman chris...
4,John Carter,taylor kitsch lynn collins samantha morton and...


In [15]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [16]:
recommend_movies("Avatar", 5)


NameError: name 'recommend_movies' is not defined

In [17]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def recommend_movies(title, top_n=5):
    if title not in indices:
        return "Movie not found in dataset"
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return movies[['title']].iloc[movie_indices]


In [18]:
recommend_movies("Avatar", 5)

,title
2403,Aliens
1804,Snow White: A Tale of Terror
94,Guardians of the Galaxy
2060,Out of the Furnace
2361,The Ice Storm


In [19]:
movies['content'] = (
    movies['actors'] + ' ' +
    movies['actors'] + ' ' +   # actors weight ×2
    movies['director'] + ' ' +
    movies['director'] + ' ' +
    movies['director']         # director weight ×3
).str.lower()


In [20]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [21]:
recommend_movies("Avatar", 5)

,title
2403,Aliens
94,Guardians of the Galaxy
279,Terminator 2: Judgment Day
3439,The Terminator
25,Titanic


In [22]:
movies.columns

Index(['movie_id', 'title', 'cast', 'crew', 'content', 'actors', 'director'], dtype='str')

In [23]:
movies['genres_clean'] = (
    movies['genres']
    .str.replace('|', ' ', regex=False)
    .str.lower()
)

KeyError: 'genres'

In [24]:
import requests


In [25]:
API_KEY = "YOUR_TMDB_API_KEY"
def get_genre_mapping():
    url = "https://api.themoviedb.org/3/genre/movie/list"
    params = {"api_key": API_KEY}
    response = requests.get(url, params=params).json()
    return {g['id']: g['name'].lower() for g in response['genres']}

genre_map = get_genre_mapping()


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [26]:
import pandas as pd
import ast

tmdb = pd.read_csv("tmdb_5000_movies.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'tmdb_5000_movies.csv'

In [26]:
import pandas as pd
import ast

tmdb = pd.read_csv("tmdb_5000_movies.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'tmdb_5000_movies.csv'

In [27]:
import pandas as pd
import ast

tmdb = pd.read_csv("tmdb_5000_movies.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'tmdb_5000_movies.csv'

In [28]:
tmdb = pd.read_csv("../data/tmdb_5000_movies.csv")

In [29]:
tmdb.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [30]:
import os
os.getcwd()

'C:\\Users\\VSB-AIDSPC15\\Movie-Recommendation-Engine\\notebooks'

In [31]:
import ast

def extract_genres(text):
    try:
        genres = ast.literal_eval(text)
        return ' '.join([g['name'].lower() for g in genres])
    except:
        return ''

tmdb['genres_clean'] = tmdb['genres'].apply(extract_genres)
tmdb[['title', 'genres_clean']].head()

,title,genres_clean
0,Avatar,action adventure fantasy science fiction
1,Pirates of the Caribbean: At World's End,adventure fantasy action
2,Spectre,action adventure crime
3,The Dark Knight Rises,action crime drama thriller
4,John Carter,action adventure science fiction


In [32]:
movies = movies.merge(
    tmdb[['title', 'genres_clean']],
    on='title',
    how='left'
)

movies['genres_clean'] = movies['genres_clean'].fillna('')


In [33]:
movies[['title', 'genres_clean']].head()

,title,genres_clean
0,Avatar,action adventure fantasy science fiction
1,Pirates of the Caribbean: At World's End,adventure fantasy action
2,Spectre,action adventure crime
3,The Dark Knight Rises,action crime drama thriller
4,John Carter,action adventure science fiction


In [34]:
movies['hybrid_content'] = (
    movies['actors'] + ' ' +
    movies['actors'] + ' ' +        # actors ×2
    movies['director'] + ' ' +
    movies['director'] + ' ' +
    movies['director'] + ' ' +      # director ×3
    movies['genres_clean'] + ' ' +
    movies['genres_clean']           # genres ×2
).str.lower()

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english', max_features=8000)
tfidf_matrix = tfidf.fit_transform(movies['hybrid_content'])

cosine_sim = cosine_similarity(tfidf_matrix)

In [36]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def recommend_movies(title, top_n=5):
    if title not in indices:
        return "Movie not found"
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return movies[['title', 'genres_clean', 'director']].iloc[movie_indices]

In [37]:
recommend_movies("Avatar", 5)

,title,genres_clean,director
2405,Aliens,horror action thriller science fiction,James Cameron
94,Guardians of the Galaxy,action science fiction adventure,James Gunn
279,Terminator 2: Judgment Day,action thriller science fiction,James Cameron
587,The Abyss,adventure action thriller science fiction,James Cameron
3442,The Terminator,action thriller science fiction,James Cameron


In [38]:
def get_relevant_movies(idx):
    target_genres = set(movies.iloc[idx]['genres_clean'].split())
    return movies[
        movies['genres_clean'].apply(
            lambda x: len(target_genres & set(x.split())) > 0
        )
    ].index.tolist()

def precision_at_k(title, k=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:k+1]
    recommended = [i[0] for i in sim_scores]
    
    relevant = get_relevant_movies(idx)
    hits = len(set(recommended) & set(relevant))
    
    return hits / k

In [39]:
precision_at_k("Avatar", 5)

1.0